# v1_0 Flop C-Bet Strategy by Board Texture 

## Table of Contents

1. [Introduction](#Introduction)
1. [Load and Validate Data](#Dataset-Inspection)  
2. [Pool Overview](#Pool-Overview)  
3. [Texture Prevalence](#Texture-Prevalence)  
4. [Main Strategy Analysis](#Main-Strategy-Analysis)  


## Introduction

This study examines flop continuation-bet strategy in single-raised pots (SRPs) using hand-history data stored in a PokerTracker 4 PostgreSQL database. The focus of version 1 is to analyze how flop c-bet behavior varies by board texture across pooled 2nl, 5nl, and 10nl data.

The main strategy question is split into two related components: how often the pool continuation-bets on a given texture, and how the pool distributes its c-bet sizings once it chooses to bet. Board texture is defined using pairedness and suitedness, while the final grouped strategy outputs are segmented by blind level, relative position, and heads-up versus multiway flop context.

In addition to the main strategy table, the analysis also includes supporting pool-overview and texture-frequency outputs to provide context for sample composition and the practical weight of each texture class. Together, these outputs are intended to establish a clean first research layer that can later be extended with refined texture definitions, additional preflop action trees, and later-street analysis.

Inputs: 
- 01_pool_overview
- 02_flop_cbet_strategy_by_texture
- 03_texture_frequency

In [ ]:
import sys
print(sys.executable)

## Import Required Libraries

In [ ]:
# Import Packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


## Dataset Inspection 

This step inspects the data we are working with to ensure our database queries produced the expected outputs, as well as the dimensions of the dataset. 

In [ ]:
# Load Datasets

pool_overview = pd.read_csv("../../results/v1_0/01_pool_overview.csv")
cbet_strategy = pd.read_csv("../../results/v1_0/02_flop_cbet_strategy_by_texture.csv")
texture_freq = pd.read_csv("../../results/v1_0/03_texture_frequency.csv")

In [ ]:
# Verify Data

for name, df in {
    "pool_overview": pool_overview,
    "cbet_strategy": cbet_strategy,
    "texture_freq": texture_freq,
}.items():
    print(name, df.shape)
    print(df.head(), "\n")

## Pool Overview

This section describes the analyzed player pool by stake and preflop action, including hand counts, flop-seen rates, average player to flop, and heads-up (HU) vs multi-way (MW) flop composition. 

In [ ]:
# Inspect pool_overview dataset
pool_overview.head()

In [ ]:
# Check dataset dimensions
pool_overview.shape

In [ ]:
# Check data types
pool_overview.dtypes

In [ ]:
# Display percentages in a more readable format (e.g., 0.25 -> 25.00%)

pool_overview_display = pool_overview.copy()

pct_cols = ["pct_hands_see_flop", "pct_flop_hu", "pct_flop_mw"]
for col in pct_cols:
    pool_overview_display[col] = (pool_overview_display[col] * 100).round(2)

pool_overview_display.head()

In [ ]:
pool_all = pool_overview_display[pool_overview_display["pf_action"] == "All"].copy()
pool_all

In [ ]:
# Compare frequencies across different bet sizes for each preflop action category
pf_order = {"SRP": 0, "3BP": 1, "4BP+": 2}

pool_breakdown = (
    pool_overview_display[pool_overview_display["pf_action"] != "All"]
    .assign(pf_sort=lambda df: df["pf_action"].map(pf_order))
    .sort_values(["pf_sort", "bb_size"])
    .drop(columns="pf_sort")
    .reset_index(drop=True)
)

pool_breakdown

In [ ]:
pool_pf = pool_overview[pool_overview["pf_action"] != "All"].copy()

pivot_pf = pool_pf.pivot(
    index="bb_size",
    columns="pf_action",
    values="number_hands"
)

pivot_pf_pct = pivot_pf.div(pivot_pf.sum(axis=1), axis=0) * 100

pivot_pf_pct.plot(kind="bar", figsize=(10, 6))
plt.title("Preflop Action Distribution by Stake")
plt.xlabel("bb_size")
plt.ylabel("frequency (%)")
plt.xticks(rotation=0)
plt.legend(title="pf_action")
plt.show()

In [ ]:
# Plot the percentage of hands that see the flop by stake and preflop action
pivot_flop_seen = pool_overview.pivot(
    index="bb_size",
    columns="pf_action",
    values="pct_hands_see_flop"
)

pivot_flop_seen.plot(kind="bar", figsize=(10, 6))
plt.ylabel("pct_hands_see_flop")
plt.title("Flop-Seen Rate by Stake and Preflop Action")
plt.legend(title="pf_action")
plt.show()

In [ ]:
# Plot the percentage of hands that see the flop by stake and preflop action, separating HU and MW

pivot_hu = pool_overview.pivot(index="bb_size", columns="pf_action", values="pct_flop_hu")
pivot_mw = pool_overview.pivot(index="bb_size", columns="pf_action", values="pct_flop_mw")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot_hu.plot(kind="bar", ax=axes[0], legend=False)
axes[0].set_title("Pct Flop HU")
axes[0].set_ylabel("pct_flop_hu")
axes[0].tick_params(axis="x", rotation=0)

pivot_mw.plot(kind="bar", ax=axes[1], legend=False)
axes[1].set_title("Pct Flop MW")
axes[1].set_ylabel("pct_flop_mw")
axes[1].tick_params(axis="x", rotation=0)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="pf_action",
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=4,
    frameon=False
)

fig.suptitle("Flop HU vs MW Rate by Stake and Preflop Action", y=1.08)
plt.show()

## Texture Prevalence 

This section examines how often each board texture, defined by pairedness and suitedness, occurs within the SRP flop c-bet opportunity sample. Texture prevalence provides important context for the main strategy results by showing how much practical weight each texture class carried in the unerlying data. 

In [ ]:
# Inspect cbet_strategy dataset
texture_freq.head()

In [ ]:
# Check dataset dimensions
texture_freq.shape

In [ ]:
# Check data types
texture_freq.dtypes

In [ ]:
# Define sorting orders for categorical variables and convert frequencies to percentages 

paired_order = {"Unpaired": 0, "Paired": 1, "Trips": 2}
suited_order = {"Rainbow": 0, "Two-Tone": 1, "Monotone": 2}
num_players_order = {"HU": 0, "MW": 1}

texture_freq_display = texture_freq.copy()
texture_freq_display["texture_frequency"] = (texture_freq_display["texture_frequency"] * 100).round(2)

texture_freq_display = (
    texture_freq_display
    .assign(
        paired_sort=lambda df: df["pairedness"].map(paired_order),
        suited_sort=lambda df: df["suitedness"].map(suited_order),
        players_sort=lambda df: df["num_players"].map(num_players_order)
    )
    .sort_values(["bb_size", "players_sort", "paired_sort", "suited_sort"])
    .drop(columns=["paired_sort", "suited_sort", "players_sort"])
    .reset_index(drop=True)
)

texture_freq_display

In [ ]:
# Aggregate frequencies by bb_size, pairedness, and suitedness to get total occurrences for each texture category

texture_freq_bb = (
    texture_freq
    .groupby(["bb_size", "pairedness", "suitedness"], as_index=False)["occurrences"]
    .sum()
)

texture_freq_bb["total_flops"] = texture_freq_bb.groupby("bb_size")["occurrences"].transform("sum")
texture_freq_bb["texture_frequency"] = (
    texture_freq_bb["occurrences"] / texture_freq_bb["total_flops"]
)

texture_freq_bb.head()

In [ ]:
# Visualize the distribution of texture frequencies across different bet sizes using heatmaps

stakes = sorted(texture_freq_bb["bb_size"].unique())

fig, axes = plt.subplots(1, len(stakes), figsize=(15, 4))

for ax, stake in zip(axes, stakes):
    heatmap_df = (
        texture_freq_bb[texture_freq_bb["bb_size"] == stake]
        .pivot(index="pairedness", columns="suitedness", values="texture_frequency")
    )
    
    sns.heatmap(heatmap_df, annot=True, fmt=".3f", cmap="magma",  ax=ax) 
    ax.set_title(f"bb_size = {stake:.2f}")

plt.tight_layout()
plt.show()

## Main Strategy Analysis
This section analyzes flop c-bet strategy by board texture using the grouped strategy output. Strategy is decomposed into two parts: c-bet frequency and conditional size selection given that a c-bet occurs.

In [ ]:
# Inspect cbet_strategy dataset
cbet_strategy.head()

In [ ]:
# Check dataset dimensions
cbet_strategy.shape

In [ ]:
# Check data types
cbet_strategy.dtypes

In [ ]:
# Return True if cbet_freq between 0 and 1, otherwise return False
((cbet_strategy["cbet_frequency"] < 0) | (cbet_strategy["cbet_frequency"] > 1)).sum() == 0

In [ ]:
# size mix should sum to about 1 when cbets > 0
cbet_strategy["size_sum"] = (
    cbet_strategy["small_pct"]
    + cbet_strategy["medium_pct"]
    + cbet_strategy["large_pct"]
    + cbet_strategy["overbet_pct"]
)

cbet_strategy.loc[cbet_strategy["cbets"] > 0, ["size_sum"]].describe()

In [ ]:
strategy_all = cbet_strategy.copy()

# keep texture if already present, otherwise create it
if "texture" not in strategy_all.columns:
    strategy_all["texture"] = (
        strategy_all["pairedness"] + " | " + strategy_all["suitedness"]
    )

# convert pct columns to weighted counts using your actual column names
strategy_all["small_count"]   = (strategy_all["small_pct"]   / 100) * strategy_all["cbets"]
strategy_all["medium_count"]  = (strategy_all["medium_pct"]  / 100) * strategy_all["cbets"]
strategy_all["large_count"]   = (strategy_all["large_pct"]   / 100) * strategy_all["cbets"]
strategy_all["overbet_count"] = (strategy_all["overbet_pct"] / 100) * strategy_all["cbets"]

strategy_agg = (
    strategy_all
    .groupby(["bb_size", "texture"], as_index=False)
    .agg(
        cbet_opportunities=("cbet_opportunities", "sum"),
        cbets=("cbets", "sum"),
        small_count=("small_count", "sum"),
        medium_count=("medium_count", "sum"),
        large_count=("large_count", "sum"),
        overbet_count=("overbet_count", "sum")
    )
)

# recompute weighted strategy metrics
strategy_agg["cbet_frequency"] = (
    strategy_agg["cbets"] / strategy_agg["cbet_opportunities"] * 100
)

strategy_agg["small_pct"] = (
    strategy_agg["small_count"] / strategy_agg["cbets"] * 100
)
strategy_agg["medium_pct"] = (
    strategy_agg["medium_count"] / strategy_agg["cbets"] * 100
)
strategy_agg["large_pct"] = (
    strategy_agg["large_count"] / strategy_agg["cbets"] * 100
)
strategy_agg["overbet_pct"] = (
    strategy_agg["overbet_count"] / strategy_agg["cbets"] * 100
)

strategy_agg["size_sum"] = (
    strategy_agg["small_pct"]
    + strategy_agg["medium_pct"]
    + strategy_agg["large_pct"]
    + strategy_agg["overbet_pct"]
)

strategy_agg.head()

In [ ]:
# Visualize the distribution of cbet frequencies across different textures and bet sizes using bar charts

stakes = sorted(strategy_agg["bb_size"].unique())

fig, axes = plt.subplots(1, len(stakes), figsize=(18, 5), sharey=True)

for ax, stake in zip(axes, stakes):
    sub = strategy_agg[strategy_agg["bb_size"] == stake].copy()
    sub = sub.sort_values("cbet_frequency", ascending=False)

    ax.bar(sub["texture"], sub["cbet_frequency"])
    ax.set_title(f"bb_size = {stake:.2f}")
    ax.set_xlabel("texture")
    ax.set_ylabel("cbet_frequency")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Flop C-bet Frequency by Texture", y=1.02)
fig.subplots_adjust(wspace=0.15, bottom=0.30)
plt.show()

In [ ]:
# Visualize the distribution of cbet frequencies across different textures and bet sizes using stacked bar charts to show size mix

fig, axes = plt.subplots(1, len(stakes), figsize=(18, 5), sharey=True)

for ax, stake in zip(axes, stakes):
    sub = strategy_agg[strategy_agg["bb_size"] == stake].copy()
    # sub = sub.sort_values("cbet_frequency", ascending=False)

    x = np.arange(len(sub))
    small   = sub["small_pct"]
    medium  = sub["medium_pct"]
    large   = sub["large_pct"]
    overbet = sub["overbet_pct"]

    ax.bar(x, small, label="Small")
    ax.bar(x, medium, bottom=small, label="Medium")
    ax.bar(x, large, bottom=small + medium, label="Large")
    ax.bar(x, overbet, bottom=small + medium + large, label="Overbet")

    ax.set_title(f"bb_size = {stake:.2f}")
    ax.set_xlabel("texture")
    ax.set_ylabel("size_mix_pct")
    ax.set_xticks(x)
    ax.set_xticklabels(sub["texture"], rotation=45, ha="right")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.98), ncol=4, frameon=False)
fig.suptitle("Flop C-bet Size Mix by Texture", y=1.02)
fig.subplots_adjust(wspace=0.15, bottom=0.30, top=0.72)
plt.show()

In [ ]:
strategy_agg.sort_values(["bb_size", "cbet_frequency"], ascending=[True, False])